In [1]:
# ¿Cómo compartimos perfiles de clientes sin permitir su reidentificación mediante cuasi-identificadores?

from pathlib import Path
import hashlib
import hmac
import sqlite3

import pandas as pd


def find_activity_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / 'data').is_dir() and (candidate / 'submission').is_dir():
            return candidate
    raise FileNotFoundError('No se encontró la raíz de la actividad.')


ROOT = find_activity_root()
SECRET = b'clave-secreta-del-programa'


In [2]:
# SQLite permite expresar las reglas de transformación y conservar el caso en una base local.

raw = pd.read_csv(ROOT / 'data' / 'raw.csv')
auxiliary = pd.read_csv(ROOT / 'data' / 'auxiliary.csv')

database_path = ROOT / 'temp' / 'anonimizacion.db'
connection = sqlite3.connect(database_path)

raw.reset_index(names='row_id').to_sql('raw_customers', connection, if_exists='replace', index=False)
auxiliary.to_sql('auxiliary_customers', connection, if_exists='replace', index=False)


def pseudonymize(document_id):
    digest = hmac.new(SECRET, str(document_id).encode(), hashlib.sha256).hexdigest()
    return f'CUST-{digest[:12].upper()}'


connection.create_function('pseudonymize', 1, pseudonymize)


In [3]:
# Las reglas suprimen identificadores directos y generalizan los atributos que facilitan ataques por enlace.

connection.execute('DROP TABLE IF EXISTS anonymized_customers')
connection.execute(
    '''
    CREATE TABLE anonymized_customers AS
    SELECT
        printf('********%04d', CAST(loyalty_card_number AS INTEGER) % 10000) AS loyalty_card_number,
        annual_spend,
        pseudonymize(document_id) AS customer_id,
        CASE
            WHEN age BETWEEN 20 AND 29 THEN '20-29'
            WHEN age BETWEEN 30 AND 39 THEN '30-39'
            WHEN age BETWEEN 40 AND 49 THEN '40-49'
            WHEN age BETWEEN 50 AND 59 THEN '50-59'
            WHEN age BETWEEN 60 AND 69 THEN '60-69'
        END AS age_group,
        CASE
            WHEN city IN ('Medellín', 'Bello', 'Envigado', 'Itagüí', 'Rionegro', 'Bogotá', 'Manizales', 'Pereira', 'Bucaramanga') THEN 'Andina'
            WHEN city IN ('Barranquilla', 'Cartagena') THEN 'Caribe'
            WHEN city = 'Cali' THEN 'Pacífica'
        END AS region,
        CASE
            WHEN occupation IN ('Administradora', 'Abogada', 'Analista financiera', 'Contadora') THEN 'Servicios profesionales'
            WHEN occupation IN ('Arquitecta', 'Diseñadora gráfica', 'Ingeniero de sistemas') THEN 'Tecnología y diseño'
            WHEN occupation IN ('Médico', 'Enfermera', 'Docente') THEN 'Salud y educación'
            WHEN occupation IN ('Comerciante', 'Técnico electricista') THEN 'Comercio y oficios'
        END AS occupation_group,
        row_id
    FROM raw_customers
    '''
)
connection.commit()


In [4]:
# La consulta convierte los datos auxiliares a los mismos grupos para comprobar el riesgo residual de enlace.

attack_candidates = pd.read_sql_query(
    '''
    WITH auxiliary_groups AS (
        SELECT
            name,
            CASE
                WHEN age BETWEEN 20 AND 29 THEN '20-29'
                WHEN age BETWEEN 30 AND 39 THEN '30-39'
                WHEN age BETWEEN 40 AND 49 THEN '40-49'
                WHEN age BETWEEN 50 AND 59 THEN '50-59'
                WHEN age BETWEEN 60 AND 69 THEN '60-69'
            END AS age_group,
            CASE
                WHEN city IN ('Medellín', 'Bello', 'Envigado', 'Itagüí', 'Rionegro', 'Bogotá', 'Manizales', 'Pereira', 'Bucaramanga') THEN 'Andina'
                WHEN city IN ('Barranquilla', 'Cartagena') THEN 'Caribe'
                WHEN city = 'Cali' THEN 'Pacífica'
            END AS region,
            CASE
                WHEN occupation IN ('Administradora', 'Abogada', 'Analista financiera', 'Contadora') THEN 'Servicios profesionales'
                WHEN occupation IN ('Arquitecta', 'Diseñadora gráfica', 'Ingeniero de sistemas') THEN 'Tecnología y diseño'
                WHEN occupation IN ('Médico', 'Enfermera', 'Docente') THEN 'Salud y educación'
                WHEN occupation IN ('Comerciante', 'Técnico electricista') THEN 'Comercio y oficios'
            END AS occupation_group
        FROM auxiliary_customers
    )
    SELECT a.name, COUNT(*) AS candidate_records
    FROM auxiliary_groups a
    JOIN anonymized_customers n USING (age_group, region, occupation_group)
    GROUP BY a.name
    ORDER BY candidate_records, a.name
    ''',
    connection,
)
attack_candidates.head()


,name,candidate_records
0,Alejandro Martínez,3
1,Isabel Díaz,4
2,Diana Correa,5
3,Felipe Vélez,5
4,Carolina Álvarez,6


In [5]:
# La entrega contiene solamente la tabla anonimizada que puede compartirse para análisis.

anonymized = pd.read_sql_query(
    '''
    SELECT loyalty_card_number, annual_spend, customer_id, age_group, region, occupation_group
    FROM anonymized_customers
    ORDER BY row_id
    ''',
    connection,
)
anonymized.to_csv(ROOT / 'submission' / 'anonymized.csv', index=False)
connection.close()
